# Сравнение diarization: pyannote vs reference_raw

Этот ноутбук:
1. Читает JSONL-файл с `reference_raw` и `prediction_chunks`
2. Преобразует `reference_raw` в сегменты diarization
3. Приводит оба набора сегментов к RTTM
4. Считает простую overlap-метрику
5. При наличии `pyannote.metrics` считает DER


In [1]:
from pathlib import Path
import json
import re
from collections import defaultdict

# -----------------------------
# Выбор JSONL-датасета из artifacts

# 1) Укажи RUN_NAME, если хочешь конкретный прогон (например, 20260323...)
# 2) Укажи JSONL_FILENAME, если хочешь конкретную модель/файл
# -----------------------------
ARTIFACTS_ROOT = Path("kaggle/working/artifacts")
RUN_NAME = "asr_benchmark_20260406_171149"  # None -> взять последний доступный
JSONL_FILENAME = "gigaam_v3_e2e_ctc.jsonl"  # например: "whisper_large_v3.jsonl"


def pick_latest_run_dir(artifacts_root: Path) -> Path:
    if not artifacts_root.exists():
        raise FileNotFoundError(f"Не найден artifacts root: {artifacts_root.resolve()}")
    candidates = [
        p
        for p in artifacts_root.iterdir()
        if p.is_dir() and p.name.startswith("asr_benchmark_")
    ]
    if not candidates:
        raise FileNotFoundError(
            f"В {artifacts_root.resolve()} нет папок asr_benchmark_* "
        )
    # Имя run-а включает timestamp, поэтому сортировка по имени даёт хронологию
    return sorted(candidates, key=lambda p: p.name)[-1]


def list_jsonl_files(run_dir: Path) -> list[Path]:
    json_dir = run_dir / "predictions_json"
    if not json_dir.exists():
        return []
    return sorted(json_dir.glob("*.jsonl"), key=lambda p: p.name)


run_dir = (
    (ARTIFACTS_ROOT / RUN_NAME) if RUN_NAME else pick_latest_run_dir(ARTIFACTS_ROOT)
)
jsonl_files = list_jsonl_files(run_dir)

if JSONL_FILENAME:
    PATH = run_dir / "predictions_json" / JSONL_FILENAME
elif jsonl_files:
    PATH = jsonl_files[0]
else:
    PATH = None

print("ARTIFACTS_ROOT        :", ARTIFACTS_ROOT.resolve())
print("RUN_DIR              :", run_dir.resolve(), "exists=", run_dir.exists())
print(
    "PREDICTIONS_JSON_DIR :",
    (run_dir / "predictions_json").resolve(),
    "exists=",
    (run_dir / "predictions_json").exists(),
)
print("AVAILABLE_JSONL      :", [p.name for p in jsonl_files])
print("SELECTED_PATH        :", None if PATH is None else str(PATH))

if PATH is None:
    raise FileNotFoundError(
        "Не найдено ни одного .jsonl в predictions_json. "
        "Сначала сгенерируй экспорты в kaggle/working/artifacts/<run>/predictions_json "
        "(например, через ноутбук asr-benchmark-comparison-diarization-json.ipynb)."
    )
if not PATH.exists():
    raise FileNotFoundError(f"JSONL не найден: {PATH.resolve()}")

PATH.exists(), PATH

ARTIFACTS_ROOT        : D:\project\multicriteria-dialog-audit-ml\kaggle\working\artifacts
RUN_DIR              : D:\project\multicriteria-dialog-audit-ml\kaggle\working\artifacts\asr_benchmark_20260406_171149 exists= True
PREDICTIONS_JSON_DIR : D:\project\multicriteria-dialog-audit-ml\kaggle\working\artifacts\asr_benchmark_20260406_171149\predictions_json exists= True
AVAILABLE_JSONL      : ['gigaam_v3_e2e_ctc.jsonl']
SELECTED_PATH        : kaggle\working\artifacts\asr_benchmark_20260406_171149\predictions_json\gigaam_v3_e2e_ctc.jsonl


(True,
 WindowsPath('kaggle/working/artifacts/asr_benchmark_20260406_171149/predictions_json/gigaam_v3_e2e_ctc.jsonl'))

In [2]:
def load_first_row(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
        return json.loads(first_line)


def parse_reference_raw(reference_raw: str, audio_duration_s: float):
    """
    Преобразует reference_raw формата:
    (00:11) Спикер 1: ...
    (00:14) Спикер 2: ...
    в список сегментов:
    [{"start": 11, "end": 14, "speaker": "REF_1", "text": "..."}]
    """
    time_line_re = re.compile(r"^\((\d{2}):(\d{2})\)\s*(.*)$", re.M)
    matches = list(time_line_re.finditer(reference_raw))

    segments = []
    for i, m in enumerate(matches):
        mm, ss, payload = m.groups()
        start = int(mm) * 60 + int(ss)

        if i + 1 < len(matches):
            mm2, ss2, _ = matches[i + 1].groups()
            end = int(mm2) * 60 + int(ss2)
        else:
            end = float(audio_duration_s)

        payload = payload.strip()
        spk_match = re.match(r"Спикер\s*(\d+):\s*(.*)", payload)
        if spk_match:
            spk_id, text = spk_match.groups()
            segments.append(
                {
                    "start": float(start),
                    "end": float(end),
                    "speaker": f"REF_{spk_id}",
                    "text": text.strip(),
                }
            )
        else:
            # например, [рингтон] — можно игнорировать
            pass

    return segments


def normalize_prediction_chunks(prediction_chunks):
    out = []
    for chunk in prediction_chunks:
        out.append(
            {
                "start": float(chunk["start"]),
                "end": float(chunk["end"]),
                "speaker": str(chunk["speaker"]),
                "text": str(chunk.get("text", "")).strip(),
            }
        )
    return out


def overlap(a_start, a_end, b_start, b_end):
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def build_speaker_mapping(ref_segments, pred_segments):
    overlap_table = defaultdict(float)

    for pred in pred_segments:
        for ref in ref_segments:
            ov = overlap(pred["start"], pred["end"], ref["start"], ref["end"])
            if ov > 0:
                overlap_table[(pred["speaker"], ref["speaker"])] += ov

    mapping = {}
    by_pred = defaultdict(list)
    for (pred_spk, ref_spk), ov in overlap_table.items():
        by_pred[pred_spk].append((ref_spk, ov))

    for pred_spk, candidates in by_pred.items():
        best_ref = max(candidates, key=lambda x: x[1])[0]
        mapping[pred_spk] = best_ref

    return mapping, dict(overlap_table)


def evaluate_simple_overlap(ref_segments, pred_segments, mapping):
    correct = 0.0
    wrong = 0.0

    for pred in pred_segments:
        mapped_ref_spk = mapping.get(pred["speaker"])
        for ref in ref_segments:
            ov = overlap(pred["start"], pred["end"], ref["start"], ref["end"])
            if ov <= 0:
                continue
            if mapped_ref_spk == ref["speaker"]:
                correct += ov
            else:
                wrong += ov

    total_ref = sum(r["end"] - r["start"] for r in ref_segments)

    covered = 0.0
    for ref in ref_segments:
        intervals = []
        for pred in pred_segments:
            s = max(pred["start"], ref["start"])
            e = min(pred["end"], ref["end"])
            if e > s:
                intervals.append((s, e))

        intervals.sort()
        merged = []
        for s, e in intervals:
            if not merged or s > merged[-1][1]:
                merged.append([s, e])
            else:
                merged[-1][1] = max(merged[-1][1], e)

        covered += sum(e - s for s, e in merged)

    missed = total_ref - covered

    return {
        "total_ref_speech_sec": round(total_ref, 3),
        "correct_sec": round(correct, 3),
        "wrong_sec": round(wrong, 3),
        "missed_sec": round(missed, 3),
        "coverage_ratio": round(covered / total_ref, 4) if total_ref else 0.0,
        "correct_ratio": round(correct / total_ref, 4) if total_ref else 0.0,
    }


def write_rttm(segments, uri, out_path):
    out_path = Path(out_path)
    with open(out_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = seg["start"]
            dur = seg["end"] - seg["start"]
            spk = seg["speaker"]
            line = f"SPEAKER {uri} 1 {start:.3f} {dur:.3f} <NA> <NA> {spk} <NA> <NA>\n"
            f.write(line)
    return out_path

In [3]:
row = load_first_row(PATH)


def get_first_present(d: dict, keys: list[str], default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


reference_raw = get_first_present(
    row, ["reference_raw", "reference", "transcription", "text"]
)
audio_duration_s = get_first_present(
    row, ["audio_duration_s", "duration_s", "audio_duration", "duration"]
)
prediction_chunks = get_first_present(
    row, ["prediction_chunks", "chunks", "prediction", "hypothesis_chunks"]
)

if reference_raw is None:
    raise KeyError("В JSONL нет reference_raw/reference/transcription/text")
if audio_duration_s is None:
    raise KeyError("В JSONL нет audio_duration_s/duration_s/audio_duration/duration")
if prediction_chunks is None:
    raise KeyError("В JSONL нет prediction_chunks/chunks/prediction/hypothesis_chunks")

ref_segments = parse_reference_raw(str(reference_raw), float(audio_duration_s))
pred_segments = normalize_prediction_chunks(prediction_chunks)

mapping, overlap_table = build_speaker_mapping(ref_segments, pred_segments)
metrics = evaluate_simple_overlap(ref_segments, pred_segments, mapping)

print("=== Speaker mapping ===")
for k, v in mapping.items():
    print(f"{k} -> {v}")

print("\n=== Overlap table ===")
for (pred_spk, ref_spk), ov in sorted(overlap_table.items()):
    print(f"{pred_spk} vs {ref_spk}: {ov:.3f} sec")

print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

=== Speaker mapping ===
SPEAKER_00 -> REF_2

=== Overlap table ===
SPEAKER_00 vs REF_1: 3.129 sec
SPEAKER_00 vs REF_2: 4.229 sec

=== Metrics ===
total_ref_speech_sec: 11.0
correct_sec: 4.229
wrong_sec: 3.129
missed_sec: 3.642
coverage_ratio: 0.6689
correct_ratio: 0.3844


In [4]:
# Кладём RTTM рядом с выбранным run-ом, чтобы не зависеть от /mnt/data
out_dir = run_dir / "diarization_compare"
out_dir.mkdir(parents=True, exist_ok=True)

ref_rttm = write_rttm(ref_segments, "sample_1", out_dir / "reference.rttm")
pred_rttm = write_rttm(pred_segments, "sample_1", out_dir / "prediction.rttm")

ref_rttm, pred_rttm

(WindowsPath('kaggle/working/artifacts/asr_benchmark_20260406_171149/diarization_compare/reference.rttm'),
 WindowsPath('kaggle/working/artifacts/asr_benchmark_20260406_171149/diarization_compare/prediction.rttm'))

## DER через pyannote.metrics

Если пакет не установлен, раскомментируй строку ниже:

```python
!pip install pyannote.metrics pyannote.database
```


In [5]:
try:
    from pyannote.database.util import load_rttm
    from pyannote.metrics.diarization import DiarizationErrorRate

    ref = load_rttm(ref_rttm)
    hyp = load_rttm(pred_rttm)

    metric = DiarizationErrorRate()
    der = metric(ref["sample_1"], hyp["sample_1"])
    print("DER:", der)
except Exception as e:
    print("Не удалось посчитать DER автоматически:")
    print(type(e).__name__, e)

DER: 0.6154545454545456


c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyannote\metrics\utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


In [6]:
print("Reference segments:")
for seg in ref_segments:
    print(seg)

print("\nPrediction segments (first 20):")
for seg in pred_segments[:20]:
    print(seg)

Reference segments:
{'start': 1.0, 'end': 2.0, 'speaker': 'REF_1', 'text': 'Алло'}
{'start': 2.0, 'end': 3.0, 'speaker': 'REF_2', 'text': 'Алло, добрый день'}
{'start': 3.0, 'end': 4.0, 'speaker': 'REF_1', 'text': 'Добрый'}
{'start': 4.0, 'end': 6.0, 'speaker': 'REF_2', 'text': 'А подскажите, пожалуйста, с кем я могу поговорить по поводу сотрудничества?'}
{'start': 6.0, 'end': 7.0, 'speaker': 'REF_2', 'text': 'Кто занимается этим вопросом?'}
{'start': 7.0, 'end': 9.0, 'speaker': 'REF_1', 'text': 'Ни с кем'}
{'start': 9.0, 'end': 10.0, 'speaker': 'REF_2', 'text': 'Ни с кем?'}
{'start': 10.0, 'end': 11.0, 'speaker': 'REF_1', 'text': 'Угу'}
{'start': 11.0, 'end': 12.0, 'speaker': 'REF_2', 'text': 'Почему?'}

Prediction segments (first 20):
{'start': 1.26284375, 'end': 2.7815937500000003, 'speaker': 'SPEAKER_00', 'text': 'Алло? Алло, добрый день.'}
{'start': 3.1697187500000004, 'end': 6.814718750000001, 'speaker': 'SPEAKER_00', 'text': 'Добрый день. А подскажите, пожалуйста, с кем могу пог